<a href="https://colab.research.google.com/github/peremartra/optipfair/blob/main/examples/data-driven_width_pruning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OptiPFair Notebook Series – Example: Data-Driven Width Pruning.

![optiPfair Logo](https://github.com/peremartra/optipfair/blob/main/images/optiPfair.png?raw=true)


This notebook demonstrates how to use the **data-driven width pruning** functionality introduced in OptiPFair v0.2.0. This hybrid approach combines static weight analysis using Peak-to-Peak Magnitude (PPM) with activation statistics collected during a calibration phase, providing more informed decisions about which neurons are truly important for your specific use case.

Unlike traditional static pruning that only considers weight magnitudes, data-driven pruning captures how neurons actually behave with your target data, resulting in better performance preservation after pruning.

**Reference**: Martra, P. (2025). Fragile Knowledge, Robust Instruction-Following: The Width Pruning Dichotomy in Llama-3.2. ArXiv. https://arxiv.org/abs/2512.22671

##Recommended Environment

- **Platform**: [Google Colab](https://colab.research.google.com)  
- **Hardware**: GPU runtime (recommended: T4 or better for 1B–3B models)  
- **Dependencies**: Installed automatically in the first cell (optipfair, transformers, torch)

##by Pere Martra.

- [LinkedIn](https://www.linkedin.com/in/pere-martra)  
- [GitHub](https://github.com/peremartra)  
- [X / Twitter](https://x.com/peremartra)

---

> If you find this useful, please ⭐ the [repository](https://github.com/peremartra/optipfair) and share it!
---
If you want to use your favorite LLM to create code with optiPfair, you just need to provide it with the file: [**optipfair_llm_reference_manual.txt**](https://github.com/peremartra/optipfair/blob/main/optipfair_llm_reference_manual.txt), which contains all the necessary information for the LLM to become an expert in using the library.

## 1. Installation and Setup
First, we install the necessary libraries.

In [ ]:
!pip install -q transformers optipfair torch datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.3 MB/s eta 0:00:00


In [ ]:
import torch
import os
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from torch.utils.data import DataLoader, TensorDataset
from optipfair import prune_model


# Check device availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Using device: cuda
GPU: Tesla T4
GPU Memory: 14.7 GB


## 2. Configuration Parameters

Configure the pruning parameters. These are easily adjustable to experiment with different settings.

In [ ]:
# Model configuration
MODEL_NAME = 'meta-llama/Llama-3.2-1B'

# Dataset configuration
CALIBRATION_SAMPLES = 100  # Number of samples for calibration (adjustable)
BATCH_SIZE = 8
MAX_LENGTH = 512

# Pruning configuration
PRUNING_PERCENTAGE = 20  # Percentage of neurons to remove

## 3. Load Model

For this example, we'll use the Llama-3.2-1B model. The data-driven pruning will help us identify and remove the least important neurons based on actual activation patterns from our calibration dataset.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()

# Count original parameters
original_params = sum(p.numel() for p in model.parameters())
print(f"Original model parameters: {original_params:,}")

## 4. Load & Prepare Calibration Dataset

The calibration dataset is crucial for data-driven pruning. The model will process this data to capture activation statistics, which inform the pruning decisions.

**Important**: For best results, use a dataset that represents your target domain or use case. Here we use WikiText as a general example, but you could use:
- Domain-specific texts (medical, legal, financial, etc.)
- Your actual task dataset
- A representative sample of your production data

In [ ]:
# Load dataset
dataset = load_dataset(
    'wikitext',
    'wikitext-2-raw-v1',
    split=f'train[:{CALIBRATION_SAMPLES}]'
)

print(f"Loaded {len(dataset)} samples for calibration")

In [ ]:
def prepare_calibration_dataloader(dataset, text_field='text'):
    """
    Prepare a dataloader for calibration.

    The dataloader will provide batches of tokenized text that the model
    processes during the calibration phase to capture activation statistics.
    """
    def tokenize_function(examples):
        if text_field in examples:
            texts = examples[text_field]
        elif 'sms' in examples:
            texts = examples['sms']
        else:
            texts = examples[list(examples.keys())[0]]

        return tokenizer(
            texts,
            truncation=True,
            padding='max_length',
            max_length=MAX_LENGTH,
            return_tensors='pt'
        )

    tokenized = dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=dataset.column_names
    )
    tokenized.set_format(type='torch', columns=['input_ids', 'attention_mask'])

    return DataLoader(tokenized, batch_size=BATCH_SIZE, shuffle=False)

calibration_loader = prepare_calibration_dataloader(dataset)
print(f"Calibration dataloader ready with {len(calibration_loader)} batches")

## 5. Data-Driven Width Pruning

Now we apply data-driven pruning using the `prune_model()` function. By providing a `dataloader`, the function automatically switches to **hybrid importance calculation** that combines:

1. **Static component**: Peak-to-Peak Magnitude (PPM) weight analysis
2. **Dynamic component**: Activation statistics from calibration data

This hybrid approach is based on the CFSP method ("CFSP: An Efficient Structured Pruning Framework for LLMs", arXiv:2409.13199v2).

### How it works:

**Phase 1 - Calibration**: Forward passes capture activation norms at down_proj inputs

**Phase 2 - Hybrid Importance**: Combines PPM weight magnitudes with activation statistics

**Phase 3 - Pruning**: Removes neuron pairs with lowest importance scores

In [ ]:
pruned_model, stats = prune_model(
    model=model,
    pruning_type="MLP_GLU",
    neuron_selection_method="MAW",  # PPM (Peak-to-Peak Magnitude) - Only MAW supports data-driven pruning
    pruning_percentage=PRUNING_PERCENTAGE,
    dataloader=calibration_loader,  # This triggers data-driven mode
    show_progress=True,
    return_stats=True
)

print("\n" + "="*60)
print("PRUNING RESULTS")
print("="*60)
print(f"Original parameters:     {stats['original_parameters']:,}")
print(f"Pruned parameters:       {stats['pruned_parameters']:,}")
print(f"Parameters removed:      {stats['original_parameters'] - stats['pruned_parameters']:,}")
print(f"Reduction:               {stats['percentage_reduction']:.2f}%")
print(f"Target reduction:        {PRUNING_PERCENTAGE}%")

Pruning layers: 100%|██████████| 16/16 [00:05<00:00,  3.10it/s]


PRUNING RESULTS
Original parameters:     1,235,814,400
Pruned parameters:       1,074,792,448
Parameters removed:      161,021,952
Reduction:               13.03%
Target reduction:        20%


## 6. Verify the Pruned Model

Let's verify that the pruned model works correctly by generating some text.

In [ ]:
print("Loading fresh model for static pruning comparison...")
model_static = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
model_static.eval()

# Apply static pruning using PPM (Peak-to-Peak Magnitude) - no dataloader
print("\nApplying static pruning (PPM weights only)...")
static_pruned_model, static_stats = prune_model(
    model=model_static,
    pruning_type="MLP_GLU",
    neuron_selection_method="MAW",  # PPM metric
    pruning_percentage=PRUNING_PERCENTAGE,
    dataloader=None,  # No calibration = static pruning
    show_progress=True,
    return_stats=True
)

print("\n" + "="*60)
print("STATIC PRUNING RESULTS (for comparison)")
print("="*60)
print(f"Original parameters:     {static_stats['original_parameters']:,}")
print(f"Pruned parameters:       {static_stats['pruned_parameters']:,}")
print(f"Reduction:               {static_stats['percentage_reduction']:.2f}%")
print("="*60)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Generating text with pruned model...

Prompt: The future of artificial intelligence is
------------------------------------------------------------
The future of artificial intelligence is likely to be a challenge for the future. AI will not be a simple task for us to develop. AI will be a challenge for us to develop. AI will be a challenge for us to develop. AI will be a challenge for us to develop
------------------------------------------------------------


## 7. Save the Pruned Model

Save the pruned model and tokenizer for later use.

In [ ]:
output_dir = "./pruned-llama-3.2-1B-datadriven"

pruned_model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"✅ Pruned model saved to: {output_dir}")
print(f"\nModel size reduced by {stats['percentage_reduction']:.2f}%")
print(f"Parameters: {stats['original_parameters']:,} → {stats['pruned_parameters']:,}")

✅ Pruned model saved to: ./pruned-llama-3.2-1B-datadriven

Model size reduced by 13.03%
Parameters: 1,235,814,400 → 1,074,792,448


## 8. Comparison: Static vs Data-Driven Pruning

To better understand the difference, let's also perform static pruning (without calibration data) on a fresh model and compare.

In [ ]:
# Clear memory
del model, pruned_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Load fresh model for static pruning
print("Loading fresh model for static pruning comparison...")
model_static = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
model_static.eval()

# Apply static pruning (no dataloader)
print("\nApplying static pruning (weights only)...")
static_pruned_model, static_stats = prune_model(
    model=model_static,
    pruning_type="MLP_GLU",
    neuron_selection_method="MAW",
    pruning_percentage=PRUNING_PERCENTAGE,
    dataloader=None,  # No calibration = static pruning
    show_progress=True,
    return_stats=True
)

print("\n" + "="*60)
print("STATIC PRUNING RESULTS (for comparison)")
print("="*60)
print(f"Original parameters:     {static_stats['original_parameters']:,}")
print(f"Pruned parameters:       {static_stats['pruned_parameters']:,}")
print(f"Reduction:               {static_stats['percentage_reduction']:.2f}%")
print("="*60)

Loading fresh model for static pruning comparison...

Applying static pruning (weights only)...


Pruning layers: 100%|██████████| 16/16 [00:05<00:00,  2.80it/s]


STATIC PRUNING RESULTS (for comparison)
Original parameters:     1,235,814,400
Pruned parameters:       1,074,792,448
Reduction:               13.03%


In [ ]:
# Test generation with static pruned model
test_prompt = "The future of artificial intelligence is"
inputs = tokenizer(test_prompt, return_tensors="pt").to(device)

print("Generating text with STATIC pruned model...\n")
print(f"Prompt: {test_prompt}")
print("-" * 60)

with torch.no_grad():
    outputs = static_pruned_model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)
print("-" * 60)

Generating text with STATIC pruned model...

Prompt: The future of artificial intelligence is
------------------------------------------------------------
The future of artificial intelligence is a topic that has been talked about for a long time and the reason for this is that the question of whether it is human or artificial has emerged as the main issue in the world of research. This is a very difficult issue to tackle as it is
------------------------------------------------------------


## Key Takeaways

1. **Data-Driven Pruning** uses both weights and activation statistics for better-informed pruning decisions
2. **Static Pruning** only considers weight magnitudes, which may not reflect actual neuron importance during inference
3. Both methods achieve similar parameter reduction, but data-driven pruning typically preserves model quality better
4. The calibration dataset should represent your target domain for best results
5. Data-driven pruning is especially valuable when adapting models to specific domains or tasks

## Next Steps

After pruning, you might want to:
- **Fine-tune** the pruned model on your task to recover performance
- **Knowledge Distillation** using the original model as teacher
- **Quantization** for additional size reduction
- **Evaluation** on your specific benchmarks to measure quality preservation

### ➡️ [**Star OptiPFair on GitHub**](https://github.com/peremartra/optipfair)

---
You can also follow my work and new projects on:

* **[LinkedIn](https://www.linkedin.com/in/pere-martra/)**
* **[X / Twitter](https://twitter.com/PereMartra)**